In [ ]:
#Import the packages needed for the data preprocessing

import os
import pickle
import numpy as np
from scipy.signal import resample
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

In [ ]:
#Check to see if GPU is running

print(f"PyTorch Version: {torch.version.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA is not available. You are currently running on the CPU.")

PyTorch Version: 2.5.1+cu121
Is CUDA available? True
GPU Name: NVIDIA A40
Total GPU Memory: 47.62 GB


In [ ]:
#Load the WESAD dataset files
data_path = r'/home/u716716/thesis/wesad data/WESAD'  #Change this to your own path
subjects = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

df = {}
# Open the data files and add it to a dictionary so that the data is easy to work with
for subject in subjects:
    file_path = os.path.join(data_path, subject, subject + '.pkl')

    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding = 'latin1')
    df[subject] = data

/tmp/ipykernel_1331615/63523465.py:10: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(f, encoding = 'latin1')


In [ ]:
#A function to put a filter over the BVP and EDA data, to remove unnecessary noise from the data

def apply_filter(data, signal_type):
    """" 
    Apply signal-specific filtering to physiologicla signals.

    Parameters:
    data: array-like, input physiologicla signal
    signal_type: str, Type of signal ("BVP", "EDA", "TEMP", "ACC")

    Returns: array-like filtered signal
    """

    #Sampling frequencies (Hz) for each physiological signal
    fs_map = {'BVP' : 64, 'EDA': 4, 'TEMP': 4, 'ACC' : 32}
    fs = fs_map.get(signal_type, 64)

    if signal_type == 'BVP':
        #Apply a Butterworth band-pass filter to retain frequencies between 1 and 8 Hz, which corresponds to the useful BVP range
        b, a = butter(4, [1, 8.0], btype = 'band', fs = fs)
        filtered = filtfilt(b, a, data)

        #Remove extreme outliers by clipping values outside the 1st and 99th percentiles
        lower = np.percentile(filtered, 1)
        upper = np.percentile(filtered, 99)
        return np.clip(filtered, lower, upper)
        
    
    elif signal_type == 'EDA':
        #Apply a Butterworth low-pass filter with a cutoff frequence of 1 Hz to smooth the EDA signal
        b, a, = butter(4, 1.0, btype = "low", fs = fs)
        return filtfilt(b, a, data)
    
    #Return the original data for signals without additional filtering
    return data

In [ ]:
#Define the labels for the binary classification task
#Baseline (1) is labelled as baseline (0)
#Stress (2) is labelled as stress (1)
#Amusement (3) is labelled as amusement (2)
THREECLASS_LABELS = {1: 0, 2: 1, 3:2} 

#Labels that will be used in the analysis
VALID_LABELS = [1, 2, 3]

#Select the physiological signal to preprocess
CURRENT_SIGNAL_TYPE = 'TEMP'

#Sampling frequencies (Hz) for each signal type
fs_map = {'BVP' : 64, 'ACC' : 32, 'EDA': 4, 'TEMP': 4}
fs = fs_map.get(CURRENT_SIGNAL_TYPE, 64)

processed_data = {}

#Preprocess the data for each subject
for subject in subjects:

    #Load the selected wrist signal
    raw_signal = df[subject]['signal']['wrist'][CURRENT_SIGNAL_TYPE]

    #For ACC, combine the three axes into a single magnitude signal
    if CURRENT_SIGNAL_TYPE == 'ACC':
        signal = np.sqrt(np.sum(np.square(raw_signal), axis=1))
    else:
        signal = raw_signal.flatten()

    #Apply signal-specific filtering
    filtered_signal = apply_filter(signal, CURRENT_SIGNAL_TYPE)

    #Load the labels corresponding to the recording
    labels = df[subject]['label']

    #Downsample the labels to match the signal length
    ratio = len(labels) / len(filtered_signal)
    indices = (np.arange(len(filtered_signal)) * ratio).astype(int)
    labels_ds = labels[indices]

    #Keep only baseline, stress and amusement labels
    mask = np.isin(labels_ds, VALID_LABELS)
    signal_final = filtered_signal[mask]
    labels_final = labels_ds[mask]

    #Create overlapping windows of 60 seconds with a 1-second step size
    window_size = fs * 60
    step_size = fs * 1

    X, y = [], []

    for start in range(0, len(signal_final) - window_size, step_size):
        end = start + window_size
        window_labels = labels_final[start:end]
        #Only keep windows with a single consistent label
        if len(np.unique(window_labels)) == 1:
            label = window_labels[0]
            X.append(signal_final[start:end])
            y.append(THREECLASS_LABELS[label])

    #Store the processed windows and labels for each subject
    processed_data[subject] = (np.array(X), np.array(y))

#Example output for subject S2
X_test, y_test = processed_data["S3"]

print(f"Processed {CURRENT_SIGNAL_TYPE} for S2")
print(f"Shape: {X_test.shape} | Labels: {np.bincount(y_test)}")
print(f"Subject S2 Windows: {processed_data['S3'][0].shape}")

X_S2, y_S2 = processed_data['S2']

print (f"X shape: {X_S2.shape}")
print(f"Label distribution: {np.bincount(y_S2)}")

Processed TEMP for S2
Shape: (1977, 240) | Labels: [1081  581  315]
Subject S2 Windows: (1977, 240)
X shape: (1943, 240)
Label distribution: [1085  556  302]


In [ ]:
#Save the preprocessed data as a pickle file so it can be reused later without repeating the preprocessing steps
file_path = f"processed_{CURRENT_SIGNAL_TYPE}_data_CLASS.pkl"

with open (file_path, 'wb') as f: 
    pickle.dump(processed_data, f)

#Confirm that the file was successfully created
print(f"Successfully saved {CURRENT_SIGNAL_TYPE} data to {file_path}")

Successfully saved TEMP data to processed_TEMP_data_CLASS.pkl


In [ ]:
import scipy
def load_and_combine_data():
    """
    Load the preprocessed ACC, BVP, EDA and TEMP data, resample the signals
    to a common length and combine them into a single multimodal dataset.

    For the multimodal dataset without TEMP, remove TEMP from the signals and data
    """

    #Physiological signals that will be combined
    signals = ["ACC", "BVP", "EDA", "TEMP"]
    data_dicts = {}

    #Load the preprocessed pickle files
    for sig in signals:
        filename = f"processed_{sig}_data_CLASS.pkl"
        print(f"Loading{filename}")
        with open (filename, 'rb') as f:
            data_dicts[sig] = pickle.load(f)
    
    combined_dict = {}

    subjects = data_dicts['BVP'].keys()

    #Common window length used for all signals
    TARGET_LEN = 240

    #Process each subject separately
    for sub in subjects:
        acc_feat, acc_y = data_dicts['ACC'][sub]
        bvp_feat, _ = data_dicts['BVP'][sub]
        eda_feat, _ = data_dicts['EDA'][sub]
        temp_feat, _ = data_dicts['TEMP'][sub]

        #Determine the maximum number of windows available for all signals
        min_windows = min(acc_feat.shape[0], bvp_feat.shape[0],
                          eda_feat.shape[0], temp_feat.shape[0])

        #Resample ACC and BVP to a common length
        acc_resampled = scipy.signal.resample(acc_feat[:min_windows], TARGET_LEN, axis = 1)
        bvp_resampled = scipy.signal.resample(bvp_feat[:min_windows], TARGET_LEN, axis =1)

        #EDA and TEMP already have the desired length and only needs trunction
        eda_resampled = eda_feat[:min_windows]
        temp_resampled = temp_feat[:min_windows]

        #Use the ACC labels as the target labels
        labels = acc_y[:min_windows]
        
        #Concatenate all signals into a single feature vector
        X_combined = np.concatenate([acc_resampled, bvp_resampled, eda_resampled, temp_resampled], axis=1)

        #Store the combined data for each subject
        combined_dict[sub] = (X_combined, labels)
        print(f"Subject{sub} combined. New feature shape: {X_combined.shape}")

        #Save the combined multimodal dataset
        with open('processed_COMBINED-TEMP_data_CLASS.pkl', 'wb') as f:
            pickle.dump(combined_dict, f)

        print("Done!")

if __name__ == "__main__":
    load_and_combine_data()

Loadingprocessed_ACC_data_CLASS.pkl
Loadingprocessed_BVP_data_CLASS.pkl
Loadingprocessed_EDA_data_CLASS.pkl
SubjectS2 combined. New feature shape: (1943, 720)
Done!
SubjectS3 combined. New feature shape: (1977, 720)
Done!
SubjectS4 combined. New feature shape: (1987, 720)
Done!
SubjectS5 combined. New feature shape: (2039, 720)
Done!
SubjectS6 combined. New feature shape: (2024, 720)
Done!
SubjectS7 combined. New feature shape: (2020, 720)
Done!
SubjectS8 combined. New feature shape: (2030, 720)
Done!
SubjectS9 combined. New feature shape: (2019, 720)
Done!
SubjectS10 combined. New feature shape: (2099, 720)
Done!
SubjectS11 combined. New feature shape: (2050, 720)
Done!
SubjectS13 combined. New feature shape: (2047, 720)
Done!
SubjectS14 combined. New feature shape: (2049, 720)
Done!
SubjectS15 combined. New feature shape: (2055, 720)
Done!
SubjectS16 combined. New feature shape: (2043, 720)
Done!
SubjectS17 combined. New feature shape: (2098, 720)
Done!


In [ ]:
#Import the packages needed for the CNN model

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
#Makes sure the input data has the right form for the 1D-CNN model, by changing the shape from (N, Length) to (N, 1, Length)
class WESADDataset(Dataset):
    def __init__(self, X, y):

        #Convert NumPy arrays to PyTorch tensors and add a channel dimension
        self.X = torch.from_numpy(X).float().unsqueeze(1)

        #Convert labels to PyTorch tensors
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        #Return the number of samples
        return len(self.X)
    
    def __getitem__(self, idx):
        #Return a single sample and its corresponding label
        return self.X[idx], self.y[idx]

In [ ]:
def get_loso_loaders(data_dict, test_subject, batch_size=32):
    """"
    Create training and test DataLoaders using Leave-One-Subject-Out (LOSO) cross-validation
    
    One subject is used as the test set, while the data from all remaining subjects is combined into the training set

    Parameters:
    data_dict: dict, Dictionary containing the data for each subject
    test_subject: str, Subject that will be used as the test set
    batch_size: int, optional, Number of samples per batch (default = 32)

    Returns:
    train_loader: DataLoader, DataLoader containing the training data
    test_loader: Dataloader, DataLoader containing the test data
    """
    #Separate the training and test subject
    subjects = list(data_dict.keys())
    train_subjects = [s for s in subjects if s != test_subject]

    #Combine all training subjects into one dataset
    X_train = np.concatenate([data_dict[s][0] for s in train_subjects], axis = 0)
    y_train = np.concatenate([data_dict[s][1] for s in train_subjects], axis = 0)

    #Use the selected subject as the test set
    X_test, y_test = data_dict[test_subject]

    #Convert the data to PyTorch datasets
    train_ds = WESADDataset(X_train, y_train)
    test_ds = WESADDataset(X_test, y_test)

    #Create DataLoaders for model training and evaluation
    train_loader = DataLoader(train_ds, batch_size = batch_size, shuffle = True)
    test_loader = DataLoader(test_ds, batch_size = batch_size, shuffle = False)

    return train_loader, test_loader

In [ ]:
import torch
import torch.nn as nn

class Simple1DCNN(nn.Module):
    """
    Simple 1D Convolutional Neural Network for binary stress classification
    using wrist-based physiological signals.
    """

    def __init__(self, input_length):
        super(Simple1DCNN, self).__init__()
        
        #Feature extraction through stacked 1D Convolutional layers
        self.conv_layers = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        #Compute flattened feature size after convolutions
        #Each MaxPool1d(2) halves the temporal dimension (3 pooling layers ->/8)
        self.flattened_size = (input_length // 8) * 128
        
        #Fully connected classification head
        self.fc1 = nn.Linear(self.flattened_size, 128)
        self.dropout = nn.Dropout(p=0.5)

        self.fc2 = nn.Linear(128, 3)
        self.relu = nn.ReLU()

        #Sigmoid layer is removed for the classification task

    def forward(self, x):
        #Pass through convolutional feature extractor
        x = self.conv_layers(x)

        #Flatten for fully connected layers
        x = x.view(x.size(0), -1)

        #Classification head
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

In [ ]:
#Select the best available device for model training
#Priority: CUDA(GPU) -> MPS (Apple Silicon) -> CPU

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

#Print selected device for reproducibility/debugging
print(f"Device set to: {device}")

Device set to: cuda


In [ ]:
def train_model(model, train_loader, val_loader, epochs, criterion, optimizer):
    """
    Train and validate a multiclass classification model using PyTorch

    Track loss and accuracy for both training and validation sets for each epoch.

    """
    model.to(device)

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):

        #Training phase

        model.train()
        total_train_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            #Move data to the selected device (GPU/CPU)
            inputs, labels = inputs.to(device), labels.to(device)

            #Reset gradient from the previous iteration
            optimizer.zero_grad()

            #Forward pass
            outputs = model(inputs)
            
            #Compute the training loss
            loss = criterion(outputs, labels)

            #Backpropagation and parameter update
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

            #Convert model outputs to class predictions
            _, predicted = torch.max(outputs.data, 1)

            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        
        avg_train_loss = total_train_loss / len(train_loader)
        train_accuracy = correct_train / total_train

        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)

        #Validation phase

        model.eval()
        total_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for val_inputs, val_labels in val_loader:

                # Move validation data to the selected device

                val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
                
                #Forward pass
                val_outputs = model(val_inputs)

                #Compute validation loss
                val_loss = criterion(val_outputs, val_labels)
                
                #Convert model outputs to class predictions
                _, predicted_val = torch.max(val_outputs.data, 1)

                total_val_loss += val_loss.item()
                
                total_val += val_labels.size(0)
                correct_val += (predicted_val == val_labels).sum().item()

        avg_val_loss = total_val_loss / len(val_loader)
        val_accuracy = correct_val / total_val

        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} Acc: {train_accuracy:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {val_accuracy:.4f}")

    return train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
def get_predictions(model, loader, device):
    """
    Generate classification predictions for a dataset using a trained model.

    Parameters:
    model: torch.nn.Module, Trained PyTorch model
    loader: DataLoader, DataLoader containing the dataset to evaluate
    device: torch.device, Device used for computation (CPU/GPU)

    Returns:
    all_preds: np.array, Predicted binary labels (0 or 1)
    all_labels: np.array, Ground truth labels.
    """
    model.eval()

    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:

            #Move input data to the correct device
            inputs = inputs.to(device)

            #Forward pass
            outputs = model(inputs)

            #Convert probabilities to classification predictions
            _, preds = torch.max(outputs, 1)

            #Store predictions and true labels
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    return np.array(all_preds), np.array(all_labels)

In [ ]:
#Test forward pass of the trained model using dummy input

#Batch size = 32, 1 channel, signal length = 240 (adjust to model input length)
test_input = torch.randn(32, 1, 1920).to(device) 
try:
    output = model(test_input)

    #Verify output shape (binary classification -> one output per sample)
    print(f"Success! Output shape: {output.shape}") #Expected: [32, 1]
except Exception as e:
    print(f"Still failing: {e}")

In [ ]:
#Experiment configuration
CURRENT_SIGNAL_TYPE = 'COMBINED-TEMP'

#Input length of the model (depends on preprocessing step)
INPUT_LENGTH = 720  #1920 for ACC, 3840 for BVP, 240 for EDA and TEMP

EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 0.0001

#Select device (GPU if availble, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Load the data
with open(f'processed_{CURRENT_SIGNAL_TYPE}_data_CLASS.pkl', 'rb') as f:
    data_dict = pickle.load(f)

all_loso_results = {}

#Create a directory to store the trained models
model_save_path = f"models_{CURRENT_SIGNAL_TYPE}_CLASS"
os.makedirs(model_save_path, exist_ok = True)

#Leave-One_subject-Out (LOSO) evaluation

for test_subject in data_dict.keys():

    #Create train/test splits
    train_loader, test_loader = get_loso_loaders(data_dict, test_subject, BATCH_SIZE)

    #Initialize model
    model = Simple1DCNN(input_length = INPUT_LENGTH).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE, weight_decay=1e-4)

    criterion = torch.nn.CrossEntropyLoss()

    #Train model
    t_losses, v_losses, t_accs, v_accs = train_model(model, train_loader, test_loader, EPOCHS, criterion, optimizer)
    
    #Generate predictions
    y_pred, y_true = get_predictions(model, test_loader, device)
    
    #Store results per subject
    all_loso_results[test_subject] = {
        'train_losses': t_losses,
        'val_losses': v_losses,
        'train_accs': t_accs,
        'val_accs': v_accs,
        'y_true': y_true,
        'y_pred': y_pred,
        'final_accuracy': v_accs[-1]
    }

    #Save results after each subject (safe checkpointing)
    with open(f'results_{CURRENT_SIGNAL_TYPE}_LOSO_CLASS.pkl', 'wb') as f:
        pickle.dump(all_loso_results, f)

    #Save model weights per subject
    full_path = os.path.join(model_save_path, f"model_{test_subject}.pth")

    torch.save(model.state_dict(), full_path)
    
    print(f"✓ Finished {test_subject}. Accuracy: {v_accs[-1]*100:.2f}%")


DEBUG: Model initialized with flattened_size = 11520
Epoch 1/20 | Train Loss: 0.8207 Acc: 0.6434 | Val Loss: 0.6011 Acc: 0.7952
Epoch 2/20 | Train Loss: 0.5948 Acc: 0.7573 | Val Loss: 0.5689 Acc: 0.7797
Epoch 3/20 | Train Loss: 0.4016 Acc: 0.8291 | Val Loss: 0.5924 Acc: 0.7972
Epoch 4/20 | Train Loss: 0.2884 Acc: 0.8792 | Val Loss: 0.6803 Acc: 0.7967
Epoch 5/20 | Train Loss: 0.2137 Acc: 0.9135 | Val Loss: 0.9730 Acc: 0.7926
Epoch 6/20 | Train Loss: 0.1613 Acc: 0.9400 | Val Loss: 1.0018 Acc: 0.8029
Epoch 7/20 | Train Loss: 0.1188 Acc: 0.9592 | Val Loss: 1.6740 Acc: 0.7900
Epoch 8/20 | Train Loss: 0.0861 Acc: 0.9723 | Val Loss: 1.2825 Acc: 0.7926
Epoch 9/20 | Train Loss: 0.0702 Acc: 0.9777 | Val Loss: 1.5267 Acc: 0.8029
Epoch 10/20 | Train Loss: 0.0526 Acc: 0.9842 | Val Loss: 1.7428 Acc: 0.7530
Epoch 11/20 | Train Loss: 0.0472 Acc: 0.9862 | Val Loss: 2.2455 Acc: 0.8003
Epoch 12/20 | Train Loss: 0.0363 Acc: 0.9895 | Val Loss: 2.1995 Acc: 0.8116
Epoch 13/20 | Train Loss: 0.0453 Acc: 0.9867

In [ ]:
import numpy as np

#Extract final accuracy from each LOSO fold (each test subject)
accuracies = [data['final_accuracy'] for data in all_loso_results.values()]

#Compute overall performance statistics
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

#Report LOSO performance
print(f"Mean LOSO Accuracy: {mean_acc*100:.2f}%")
print(f"Standard Deviation: {std_acc*100:.2f}%")

Mean LOSO Accuracy: 66.92%
Standard Deviation: 16.41%
